# Expanding Escher Maps with New Reactions

When working with community Escher maps, the pre-curated layout only covers a subset of
metabolism. Reactions we add — via the GUI or programmatically — land at default positions
and look terrible. This notebook demonstrates how to:

1. Load a community Escher map and inspect its structure
2. Prune reactions we don't need
3. Add new reactions from our model programmatically
4. Use NetworkX to auto-layout the new nodes while pinning existing ones
5. Use SAMMI for rapid exploration and as an alternative layout tool
6. Load the result back into Escher with flux data

## 1. Setup and model loading

In [ ]:
import cobra
from cobra.flux_analysis import pfba
import escher
import json
import copy
import networkx as nx
import numpy as np
import pandas as pd

model = cobra.io.load_model("textbook")
model.solver = "highs"

solution = pfba(model)
print(f"Model: {model.id} ({len(model.reactions)} reactions, {len(model.metabolites)} metabolites)")
print(f"Growth rate: {solution.objective_value:.4f} h⁻¹")

## 2. Load a community Escher map and inspect its structure

Escher ships with curated maps for common organisms. We'll fetch one and look at
the JSON schema — understanding this structure is essential for programmatic editing.

In [ ]:
# List available maps — look for ones matching our textbook E. coli core model
available = escher.list_available_maps()
print("Available maps:")
for m in available:
    print(f"  {m}")

In [ ]:
# Fetch the E. coli core map JSON via Escher's URL helpers.
# The map is a two-element JSON array: [header_dict, map_data_dict]
import requests

map_name = "e_coli_core.Core metabolism"
url = escher.urls.map_download_url(map_name)
print(f"Fetching: {url}")
response = requests.get(url)
response.raise_for_status()
escher_map = response.json()

header = escher_map[0]
map_data = escher_map[1]

print(f"\nHeader: {json.dumps(header, indent=2)}")
print(f"\nMap data keys: {list(map_data.keys())}")
print(f"  Reactions: {len(map_data['reactions'])}")
print(f"  Nodes:     {len(map_data['nodes'])}")

In [ ]:
# Inspect the structure of one reaction and a few node types.
# This schema is what we need to replicate when adding reactions programmatically.

sample_rxn_id = list(map_data["reactions"].keys())[0]
sample_rxn = map_data["reactions"][sample_rxn_id]
print(f"=== Sample reaction (key={sample_rxn_id}) ===")
print(json.dumps(sample_rxn, indent=2)[:2000])

# Nodes come in several types: "metabolite", "midmarker", "multimarker"
node_types = {}
for nid, node in map_data["nodes"].items():
    nt = node.get("node_type", "unknown")
    node_types.setdefault(nt, []).append(nid)

print(f"\n=== Node types ===")
for nt, ids in node_types.items():
    print(f"  {nt}: {len(ids)} nodes")
    sample = map_data["nodes"][ids[0]]
    print(f"    Example keys: {list(sample.keys())}")

## 3. Assess map coverage

Which model reactions are already on the map, and which are missing? The missing ones
are candidates for programmatic addition.

In [ ]:
# BiGG IDs on the map vs in the model
map_rxn_ids = {r["bigg_id"] for r in map_data["reactions"].values()}
model_rxn_ids = {r.id for r in model.reactions}

on_map = map_rxn_ids & model_rxn_ids
missing_from_map = model_rxn_ids - map_rxn_ids
on_map_not_in_model = map_rxn_ids - model_rxn_ids

print(f"Reactions on map AND in model:  {len(on_map)}")
print(f"In model but NOT on map:        {len(missing_from_map)}")
print(f"On map but NOT in model:        {len(on_map_not_in_model)}")

# Show the missing reactions with their fluxes — these are candidates to add
missing_df = pd.DataFrame([
    {
        "reaction": rxn_id,
        "name": model.reactions.get_by_id(rxn_id).name,
        "subsystem": model.reactions.get_by_id(rxn_id).subsystem,
        "flux": solution.fluxes[rxn_id],
    }
    for rxn_id in sorted(missing_from_map)
])
print(f"\nReactions missing from map (sorted by |flux|):")
missing_df.reindex(missing_df["flux"].abs().sort_values(ascending=False).index)

## 4. Prune unwanted reactions from the map

Remove reactions that are on the map but not in our model (or that we simply don't
care about). This is pure dict manipulation on the Escher JSON.

In [ ]:
def prune_escher_map(map_data, keep_bigg_ids):
    """Remove reactions not in keep_bigg_ids and clean up orphaned nodes.
    
    Args:
        map_data: The second element of an Escher map JSON array.
        keep_bigg_ids: Set of BiGG reaction IDs to retain.
    
    Returns:
        New map_data dict with unwanted reactions and orphaned nodes removed.
    """
    pruned = copy.deepcopy(map_data)
    
    # 1. Remove reactions whose bigg_id is not in our keep set
    rxns_to_delete = [
        rid for rid, rxn in pruned["reactions"].items()
        if rxn["bigg_id"] not in keep_bigg_ids
    ]
    for rid in rxns_to_delete:
        del pruned["reactions"][rid]
    
    # 2. Collect all node IDs still referenced by remaining reactions
    referenced_nodes = set()
    for rxn in pruned["reactions"].values():
        for seg in rxn.get("segments", {}).values():
            referenced_nodes.add(str(seg["from_node_id"]))
            referenced_nodes.add(str(seg["to_node_id"]))
    
    # 3. Remove orphaned nodes (midmarkers/multimarkers/metabolites no longer connected)
    nodes_to_delete = [
        nid for nid in pruned["nodes"]
        if nid not in referenced_nodes
        and pruned["nodes"][nid].get("node_type") != "metabolite"
    ]
    # For metabolite nodes, only remove if truly orphaned (not referenced by any segment)
    for nid in list(pruned["nodes"].keys()):
        if nid not in referenced_nodes and pruned["nodes"][nid].get("node_type") == "metabolite":
            nodes_to_delete.append(nid)
    
    for nid in nodes_to_delete:
        del pruned["nodes"][nid]
    
    return pruned, len(rxns_to_delete), len(nodes_to_delete)


# Prune: keep only reactions that exist in our model
pruned_map_data, n_rxn_removed, n_node_removed = prune_escher_map(map_data, model_rxn_ids)

print(f"Removed {n_rxn_removed} reactions and {n_node_removed} orphaned nodes")
print(f"Remaining: {len(pruned_map_data['reactions'])} reactions, {len(pruned_map_data['nodes'])} nodes")

## 5. Add new reactions to the map programmatically

Each Escher reaction needs entries in both `reactions` and `nodes`. The structure is:
- **Reaction entry**: `bigg_id`, `name`, `reversibility`, `label_x/y`, `gene_reaction_rule`,
  `metabolites` list, and `segments` (edges connecting nodes via `from_node_id`/`to_node_id`)
- **Node entries**: one `midmarker` for the reaction center, `multimarker` nodes for branch
  points, and `metabolite` nodes for each participating species

New nodes get placeholder positions — we'll fix those in step 6.

In [ ]:
# Build an index of metabolite BiGG IDs already on the map → their node IDs and positions.
# When a new reaction connects to a metabolite already on the map, we reuse that node
# instead of creating a duplicate.

existing_met_nodes = {}  # bigg_id → {node_id, x, y}
for nid, node in pruned_map_data["nodes"].items():
    if node.get("node_type") == "metabolite" and "bigg_id" in node:
        existing_met_nodes[node["bigg_id"]] = {
            "node_id": nid,
            "x": node["x"],
            "y": node["y"],
        }

print(f"Metabolite nodes already on map: {len(existing_met_nodes)}")
print("Examples:", list(existing_met_nodes.keys())[:10])

In [ ]:
def next_id(map_data, prefix="new"):
    """Generate the next available integer key for reactions or nodes."""
    all_keys = set(map_data["reactions"].keys()) | set(map_data["nodes"].keys())
    max_int = max((int(k) for k in all_keys if k.isdigit()), default=0)
    return max_int + 1


def add_reaction_to_map(map_data, cobra_rxn, existing_met_nodes, placeholder_x=0, placeholder_y=0):
    """Add a COBRApy reaction to an Escher map_data dict.
    
    New metabolite nodes that don't already exist on the map get placeholder positions.
    Existing metabolite nodes are reused by reference.
    
    Returns:
        set of newly created node IDs (these will need repositioning).
    """
    nid = next_id(map_data)
    new_node_ids = set()
    
    # Create a midmarker node for the reaction center
    mid_nid = str(nid)
    map_data["nodes"][mid_nid] = {
        "node_type": "midmarker",
        "x": placeholder_x,
        "y": placeholder_y,
    }
    new_node_ids.add(mid_nid)
    nid += 1
    
    segments = {}
    metabolites_list = []
    seg_idx = 0
    
    for met, coeff in cobra_rxn.metabolites.items():
        met_bigg = met.id
        
        # Reuse existing metabolite node or create a new one
        if met_bigg in existing_met_nodes:
            met_node_id = existing_met_nodes[met_bigg]["node_id"]
        else:
            met_node_id = str(nid)
            map_data["nodes"][met_node_id] = {
                "node_type": "metabolite",
                "x": placeholder_x + np.random.uniform(-100, 100),
                "y": placeholder_y + np.random.uniform(-100, 100),
                "bigg_id": met_bigg,
                "name": met.name,
                "label_x": placeholder_x + np.random.uniform(-100, 100),
                "label_y": placeholder_y + np.random.uniform(-100, 100),
                "node_is_primary": abs(coeff) > 0 and met_bigg not in {
                    "h_c", "h_e", "h2o_c", "h2o_e",  # common secondary metabolites
                },
            }
            existing_met_nodes[met_bigg] = {
                "node_id": met_node_id,
                "x": map_data["nodes"][met_node_id]["x"],
                "y": map_data["nodes"][met_node_id]["y"],
            }
            new_node_ids.add(met_node_id)
            nid += 1
        
        # Create a multimarker node between midmarker and metabolite
        multi_nid = str(nid)
        map_data["nodes"][multi_nid] = {
            "node_type": "multimarker",
            "x": (placeholder_x + existing_met_nodes[met_bigg]["x"]) / 2,
            "y": (placeholder_y + existing_met_nodes[met_bigg]["y"]) / 2,
        }
        new_node_ids.add(multi_nid)
        nid += 1
        
        # Segment from midmarker → multimarker
        segments[str(seg_idx)] = {
            "from_node_id": mid_nid,
            "to_node_id": multi_nid,
            "b1": None,
            "b2": None,
        }
        seg_idx += 1
        
        # Segment from multimarker → metabolite node
        segments[str(seg_idx)] = {
            "from_node_id": multi_nid,
            "to_node_id": met_node_id,
            "b1": None,
            "b2": None,
        }
        seg_idx += 1
        
        metabolites_list.append({
            "bigg_id": met_bigg,
            "coefficient": coeff,
        })
    
    # Create the reaction entry
    rxn_id = str(next_id(map_data))
    map_data["reactions"][rxn_id] = {
        "name": cobra_rxn.name,
        "bigg_id": cobra_rxn.id,
        "reversibility": cobra_rxn.lower_bound < 0,
        "label_x": placeholder_x,
        "label_y": placeholder_y - 20,
        "gene_reaction_rule": cobra_rxn.gene_reaction_rule,
        "genes": [{"bigg_id": g.id, "name": g.name} for g in cobra_rxn.genes],
        "metabolites": metabolites_list,
        "segments": segments,
    }
    
    return new_node_ids


# Select reactions to add — pick those missing from the map with nonzero flux
rxns_to_add = [
    model.reactions.get_by_id(row["reaction"])
    for _, row in missing_df.iterrows()
    if abs(row["flux"]) > 1e-6
]
# Limit to a manageable number for this demo
rxns_to_add = rxns_to_add[:8]

print(f"Adding {len(rxns_to_add)} reactions to the map:")
for r in rxns_to_add:
    print(f"  {r.id}: {r.name} (flux={solution.fluxes[r.id]:.4f})")

In [ ]:
# Add each reaction. Place new nodes at a cluster off to the side — they'll be
# repositioned by the layout algorithm in step 6.
expanded_map_data = copy.deepcopy(pruned_map_data)

# Compute the centroid of existing nodes to place new ones nearby
existing_xs = [n["x"] for n in expanded_map_data["nodes"].values() if "x" in n]
existing_ys = [n["y"] for n in expanded_map_data["nodes"].values() if "y" in n]
center_x = np.mean(existing_xs)
center_y = np.mean(existing_ys)

all_new_node_ids = set()
for i, rxn in enumerate(rxns_to_add):
    # Stagger placeholder positions so they don't all land on the same spot
    px = center_x + 400 + (i % 4) * 150
    py = center_y - 200 + (i // 4) * 300
    new_ids = add_reaction_to_map(expanded_map_data, rxn, existing_met_nodes, px, py)
    all_new_node_ids |= new_ids

print(f"Map now has {len(expanded_map_data['reactions'])} reactions, {len(expanded_map_data['nodes'])} nodes")
print(f"New node IDs to reposition: {len(all_new_node_ids)}")

## 6. Reposition new nodes with NetworkX spring layout

The key idea: extract all node positions from the Escher JSON into a NetworkX graph,
**pin** the original (well-positioned) nodes, and let `spring_layout` move only the
newly added nodes. This gives them positions that are spatially coherent with their
neighbors without disturbing the curated layout.

This is the programmatic alternative to SAMMI's interactive force-directed layout
(see section 8 for the SAMMI approach).

In [ ]:
def build_graph_from_escher(map_data):
    """Build a NetworkX graph from Escher map segments.
    
    Returns:
        G: networkx.Graph with an edge for each segment
        pos: dict of {node_id: (x, y)} from the current map positions
    """
    G = nx.Graph()
    pos = {}
    
    # Add all nodes with their current positions
    for nid, node in map_data["nodes"].items():
        if "x" in node and "y" in node:
            G.add_node(nid)
            pos[nid] = (node["x"], node["y"])
    
    # Add edges from reaction segments
    for rxn in map_data["reactions"].values():
        for seg in rxn.get("segments", {}).values():
            from_id = str(seg["from_node_id"])
            to_id = str(seg["to_node_id"])
            if from_id in pos and to_id in pos:
                G.add_edge(from_id, to_id)
    
    return G, pos


def reposition_new_nodes(map_data, new_node_ids, k=120, iterations=500):
    """Use spring_layout to reposition only new nodes, pinning existing ones.
    
    Args:
        map_data: Escher map data dict (modified in place).
        new_node_ids: Set of node ID strings that should be repositioned.
        k: Optimal distance between nodes (controls spacing).
        iterations: Number of spring layout iterations.
    
    Returns:
        The updated positions dict.
    """
    G, pos = build_graph_from_escher(map_data)
    
    # Pin all nodes that are NOT new
    fixed_nodes = [nid for nid in G.nodes() if nid not in new_node_ids]
    
    if not fixed_nodes:
        # Nothing to pin — just run unconstrained layout
        new_pos = nx.spring_layout(G, pos=pos, k=k, iterations=iterations, seed=42)
    else:
        new_pos = nx.spring_layout(
            G, pos=pos, fixed=fixed_nodes, k=k, iterations=iterations, seed=42
        )
    
    # Write computed positions back into the map data
    for nid, (x, y) in new_pos.items():
        if nid in map_data["nodes"]:
            map_data["nodes"][nid]["x"] = float(x)
            map_data["nodes"][nid]["y"] = float(y)
            # Update label positions for metabolite nodes
            if "label_x" in map_data["nodes"][nid]:
                map_data["nodes"][nid]["label_x"] = float(x)
                map_data["nodes"][nid]["label_y"] = float(y) - 20
    
    return new_pos


# Scale: spring_layout normalizes to [0,1] by default, but we want to preserve
# the Escher coordinate space. We pass the existing positions as initial positions
# and pin them, so the output stays in the same scale.

laid_out_map = copy.deepcopy(expanded_map_data)
new_pos = reposition_new_nodes(laid_out_map, all_new_node_ids, k=120, iterations=500)

print(f"Repositioned {len(all_new_node_ids)} nodes")
print(f"Total nodes in graph: {len(new_pos)}")

In [ ]:
# Visualize the layout with matplotlib to sanity-check before loading into Escher.
# Blue = original (pinned) nodes, red = newly positioned nodes.

import matplotlib.pyplot as plt

G, pos = build_graph_from_escher(laid_out_map)

fig, ax = plt.subplots(figsize=(14, 10))

original_nodes = [n for n in G.nodes() if n not in all_new_node_ids]
new_nodes = [n for n in G.nodes() if n in all_new_node_ids]

nx.draw_networkx_edges(G, pos, ax=ax, alpha=0.3, edge_color="gray")
nx.draw_networkx_nodes(G, pos, nodelist=original_nodes, ax=ax,
                       node_color="steelblue", node_size=20, alpha=0.6, label="Original")
nx.draw_networkx_nodes(G, pos, nodelist=new_nodes, ax=ax,
                       node_color="red", node_size=40, alpha=0.8, label="New")

# Label only the new metabolite nodes
new_met_labels = {
    nid: laid_out_map["nodes"][nid].get("bigg_id", "")
    for nid in new_nodes
    if laid_out_map["nodes"][nid].get("node_type") == "metabolite"
}
nx.draw_networkx_labels(G, pos, labels=new_met_labels, ax=ax, font_size=7, font_color="darkred")

ax.set_title("Escher map layout: original nodes (blue) + new reactions (red)")
ax.legend(scatterpoints=1)
ax.invert_yaxis()  # Escher uses screen coordinates (y increases downward)
plt.tight_layout()
plt.show()

## 7. Load the expanded map into Escher with flux data

Save the modified map JSON and render it in Escher. The original reactions keep their
curated positions; the new reactions have spring-layout positions.

In [ ]:
# Save the expanded map as JSON
expanded_map_json = [header, laid_out_map]
map_output_path = "expanded_core_map.json"
with open(map_output_path, "w") as f:
    json.dump(expanded_map_json, f)
print(f"Saved expanded map to {map_output_path}")

# Render in Escher with pFBA flux overlay
builder = escher.Builder(
    map_json=map_output_path,
    model=model,
    reaction_data=solution.fluxes.to_dict(),
)
builder.highlight_missing = True
builder.hide_secondary_metabolites = False

# In JupyterLab, display the interactive widget:
builder

In [ ]:
# Also save as standalone HTML for viewing outside Jupyter
builder.save_html("expanded_core_map.html")
print("Saved standalone HTML to expanded_core_map.html")

## 8. SAMMI: rapid exploration without a curated map

SAMMI takes a completely different approach — no pre-existing map needed. It generates
force-directed bipartite graphs directly from a COBRApy model, with built-in subgraph
parsing and metabolite shelving. The tradeoff: less polished than Escher, but much
faster for exploration.

SAMMI opens visualizations in the browser (not inline in Jupyter).

In [ ]:
import sammi

# --- 8a. Whole model parsed by subsystem ---
# Each subsystem gets its own force-directed subgraph.
# Shelve common cofactors so the layout focuses on primary pathway topology.

secondary_mets = [r"^h_c$", r"^h2o_c$", r"^atp_c$", r"^adp_c$", r"^pi_c$",
                  r"^nad_c$", r"^nadh_c$", r"^nadp_c$", r"^nadph_c$", r"^co2_c$"]

print("Plotting whole model by subsystem (opens in browser)...")
sammi.plot(model, "subsystem", secondaries=secondary_mets)

In [ ]:
# --- 8b. Custom subgraph: just glycolysis reactions ---
# Use sammi.parser() to specify exactly which reactions to show.

glycolysis_rxns = ["PFK", "PYK", "PGK", "GAPD", "TPI", "FBA", "PGI", "PGM", "ENO", "HEX1", "PFK"]
parser = [sammi.parser(reactions=glycolysis_rxns, name="Glycolysis")]

print("Plotting glycolysis subgraph (opens in browser)...")
sammi.plot(model, parser, secondaries=secondary_mets)

In [ ]:
# --- 8c. Flux data overlay ---
# Map pFBA fluxes onto reaction colors. Useful for comparing WT vs knockout.

flux_df = solution.fluxes.to_frame(name="pFBA_flux")
data_overlay = [sammi.data(group="reactions", kind="color", data=flux_df)]

print("Plotting glycolysis with flux overlay (opens in browser)...")
sammi.plot(model, parser, data_overlay, secondaries=secondary_mets)

## 9. SAMMI → Escher bridge

SAMMI can export its current layout as Escher-compatible JSON ("Download ESCHER" in the
GUI). This lets you use SAMMI for rapid positioning, then bring the result into Escher
for refinement. The workflow:

1. Open the SAMMI visualization in the browser
2. Adjust the force-directed layout (drag nodes, fix positions, tweak parameters)
3. Click **Upload/Download → Download ESCHER**
4. Load the exported JSON into `escher.Builder(map_json=...)` for flux overlay and polish

This is especially useful when adding reactions to an existing Escher map:
- Load the Escher map in SAMMI
- Fix the well-positioned nodes (select all → right-click → Fix position)
- The force simulation repositions only the unfixed (new) nodes
- Export back to Escher JSON

In [ ]:
# Example: loading a SAMMI-exported Escher JSON back into Escher.
# (Uncomment and update the path after exporting from the SAMMI GUI.)

# builder = escher.Builder(
#     map_json="sammi_exported_map.json",
#     model=model,
#     reaction_data=solution.fluxes.to_dict(),
# )
# builder.highlight_missing = True
# builder.hide_secondary_metabolites = True
# builder

## 10. Summary

| Approach | Best for | Tradeoffs |
|----------|----------|-----------|
| **Programmatic (this notebook, sections 2–7)** | Batch-adding reactions, reproducible pipelines, CI | No interactivity; spring_layout produces straight edges only |
| **SAMMI interactive (section 8)** | Rapid exploration, quick subnetwork visualization, positioning new nodes | Opens in browser, not inline; requires internet; less polished aesthetics |
| **SAMMI → Escher export (section 9)** | Building a starting layout for a new Escher map, fixing positions after adding reactions | Manual GUI step; export may need scale adjustment |
| **Escher GUI** | Final polish, publication figures, iterative flux comparison | Requires a pre-existing map; no auto-layout for new reactions |